<h1>Bronze Bus Live Feed<h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [3]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

<h3>Imports<h3>

In [40]:
import sys
import gc
from pyspark.sql import DataFrame as SparkDF
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
import requests
from pyspark.sql.types import FloatType
from pyspark.sql import Row
api_key = os.getenv("WARSAW_API_KEY")
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual

<h2>Bus Live Data Table<h2>

<h3>Calling Warsaw API<h3>

In [25]:
url = "https://api.um.warszawa.pl/api/action/busestrams_get/"
params = {
    "resource_id": "f2e5503e-927d-4ad3-9500-4ab9e55ebe59", 
    "apikey": api_key,
    "type": "1" # '1' to autobusy, '2' to tramwaje
}
response = requests.get(url, params={"apikey": api_key,
            "resource_id":"f2e5503e927d-4ad3-9500-4ab9e55deb59","type": "1"},
            timeout=10) #Type 1 for BUS
if response.status_code == 200:
    print("API call successful")

API call successful


<h3>Creating df<h3>

In [30]:
if response.status_code == 200:
    data = response.json()
    if isinstance(data, dict) and "result" in data and isinstance(data["result"], list):
        print(data["result"])
        df_live = spark.createDataFrame(data["result"])

[{'Lines': '225', 'Lon': 21.115383, 'VehicleNumber': '1000', 'Time': '2026-04-03 18:41:24', 'Lat': 52.234526, 'Brigade': '501'}, {'Lines': '161', 'Lon': 21.2248561, 'VehicleNumber': '1001', 'Time': '2026-04-03 21:43:52', 'Lat': 52.1866135, 'Brigade': '3'}, {'Lines': '225', 'Lon': 21.1157645, 'VehicleNumber': '1002', 'Time': '2026-04-03 18:18:08', 'Lat': 52.2345913, 'Brigade': '04'}, {'Lines': '219', 'Lon': 21.0944146, 'VehicleNumber': '1003', 'Time': '2026-04-03 21:43:53', 'Lat': 52.2212955, 'Brigade': '1'}, {'Lines': '225', 'Lon': 21.1157266, 'VehicleNumber': '1005', 'Time': '2026-04-03 17:38:36', 'Lat': 52.2345376, 'Brigade': '503'}, {'Lines': '219', 'Lon': 21.14494, 'VehicleNumber': '1006', 'Time': '2026-04-03 21:43:51', 'Lat': 52.18656, 'Brigade': '3'}, {'Lines': '108', 'Lon': 21.0237298, 'VehicleNumber': '1007', 'Time': '2026-04-03 21:43:52', 'Lat': 52.2299368, 'Brigade': '1'}, {'Lines': 'L-8', 'Lon': 21.0833845, 'VehicleNumber': '10071', 'Time': '2026-04-03 21:43:52', 'Lat': 52.4

<h3>Writing to DB<h3>

In [31]:
df_live.write.jdbc(url=jdbc_url, table="bronze.live_bus_data", 
                   mode="overwrite", properties=db_properties)

<h3>Check<h3>

In [34]:
df_live_test = spark.read.jdbc(url=jdbc_url, table="bronze.live_bus_data", 
                               properties=db_properties)
print(assertDataFrameEqual(df_live_test, df_live))

None


In [41]:
spark.catalog.clearCache()
gc.collect()
spark.stop()